### Deatiled pipeline

In [3]:
%load_ext autoreload
%autoreload 2
from scripts.utils.data_loader import create_configs, load_data
from scripts.utils.preprocessing import lof_outlier_removal
from scripts.utils.post_processing import save_results, compute_fold_shap, plot_shap_summary

from scipy.stats import randint, uniform, loguniform

from sklearn.preprocessing import PowerTransformer

from imblearn import FunctionSampler

from functools import partial
from sklearn.feature_selection import SelectPercentile
from sklearn.feature_selection import mutual_info_classif

from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE

from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV, cross_validate

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

import os
import numpy as np
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
class CorrelationBasedFeatureSelection(BaseEstimator, TransformerMixin):
    def __init__(self, intercorr_threshold=0.90, target_corr_threshold=0.25):
        self.intercorr_threshold = intercorr_threshold
        self.target_corr_threshold = target_corr_threshold
        self.to_drop_intercorrelated_ = []
        self.to_drop_target_corr_ = []
        self.to_drop_ = []
        self.selected_features_ = []

    def fit(self, X, y):
        X_df = pd.DataFrame(X) if isinstance(X, np.ndarray) else X
        y_series = pd.Series(y) if isinstance(y, np.ndarray) else y
        
        corr_matrix = X_df.corr().abs()
        upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        target_corr = X_df.apply(lambda col: col.corr(y_series)).abs()
        
        drop_intercorr_set = set()
        for col in upper_tri.columns:
            for row in upper_tri.index:
                if upper_tri.loc[row, col] > self.intercorr_threshold:
                    if row not in drop_intercorr_set and col not in drop_intercorr_set:
                        if target_corr[row] >= target_corr[col]:
                            drop_intercorr_set.add(col)
                        else:
                            drop_intercorr_set.add(row)
        
        self.to_drop_intercorrelated_ = list(drop_intercorr_set)

        X_reduced = X_df.drop(columns=self.to_drop_intercorrelated_, errors='ignore')
        target_corr_reduced = target_corr.loc[X_reduced.columns]
        n_reduced = len(target_corr_reduced)
        n_keep = int(np.ceil(self.target_corr_threshold * n_reduced))

        self.selected_features_ = (
            target_corr_reduced
            .sort_values(ascending=False)
            .head(n_keep)
            .index
            .tolist()
        )

        to_drop_target_corr_ = [
            col for col in X_reduced.columns
            if col not in self.selected_features_
        ]

        self.to_drop_target_corr_ = to_drop_target_corr_
        self.to_drop_ = self.to_drop_intercorrelated_ + self.to_drop_target_corr_

        return self

    def transform(self, X):
        X_df = pd.DataFrame(X) if isinstance(X, np.ndarray) else X.copy()
        X_selected = X_df.drop(columns=self.to_drop_, errors='ignore')
        return X_selected.values if isinstance(X, np.ndarray) else X_selected

    def set_output(self, transform):
        return self

In [4]:
# Setup experiment configurations
case_idx = 8
model_name = "KNN"
feature_selector_method = "rfe"
n_dict = {
    "n_repeats": 3,
    "outer_splits": 5,
    "inner_splits": 5,
    "n_iter": 5,
    "outer_verbose": 20,
    "inner_verbose": 1,
    "outer_n_jobs": -1,
    "inner_n_jobs": 1
}

config = create_configs(case_idx, model_name, feature_selector_method, n_dict)
out_dir = f"../results_tests/{model_name}_{feature_selector_method}/sex={config['sexes_key']}/task={config['tasks_key']}"
config.update({"out_dir": out_dir})
os.makedirs(out_dir, exist_ok=True)

# Load data
X, y, groups = load_data(config)

# Define scoring metrics
scoring = {
    "roc_auc": "roc_auc",
    "balanced_accuracy": "balanced_accuracy",
    "average_precision": "average_precision",
    "f1": "f1"
}

# Build pipeline
yj_pt = PowerTransformer(method="yeo-johnson", standardize=True)

lof_sampler = FunctionSampler(
    func=lof_outlier_removal,
    kw_args={
        "contamination": 0.05,
        "n_neighbors": 20,
        "algorithm": "auto",
        "metric": "manhattan",
    },
    validate=False,
)

if config["feature_selector_method"] == "mi_based":
    feature_selector = SelectPercentile(score_func=partial(mutual_info_classif, n_neighbors=5, random_state=42))

elif config["feature_selector_method"] == "corr_based":
    feature_selector = CorrelationBasedFeatureSelection()

elif config["feature_selector_method"] == "rfe":
    feature_selector = RFE(estimator=RandomForestClassifier(n_estimators=25, random_state=42), step = 0.1)

elif config["feature_selector_method"] == "passthrough":
    feature_selector = "passthrough"

smote = SMOTE(random_state=42)

clf = KNeighborsClassifier()

steps = [
    ("yjpt", yj_pt),
    ("outlier_removal", lof_sampler),
    ("feature_selector", feature_selector),
    ("oversampling", smote),
    ("classifier", clf),
]

pipeline = ImbPipeline(steps=steps).set_output(transform="pandas")

# Param distributions for RandomizedSearchCV
param_distributions = {}
if feature_selector_method == "mi_based":
    param_distributions.update({
        "feature_selector__percentile": randint(50, 91),  # [50, 90]
    })

elif feature_selector_method == "corr_based":
    param_distributions.update({
        "feature_selector__intercorr_threshold": uniform(0.85, 0.1),  # [0.85, 0.95]
        "feature_selector__target_corr_threshold": uniform(0.2, 0.1),  # [0.2, 0.3]
    })

elif feature_selector_method == "rfe":
    param_distributions.update({
        "feature_selector__n_features_to_select": uniform(0.1, 0.9),  # [0.1, 1.0]
    })

param_distributions.update({
    "oversampling__k_neighbors": randint(3, 8),  # [3, 7]
    "classifier__n_neighbors": list(range(3, 22, 2)),  # [3, 5, 7, ..., 21]
    "classifier__weights": ["uniform", "distance"],
    "classifier__algorithm": ["ball_tree", "kd_tree", "brute"],
    "classifier__p": [1, 2],
})

# Setup cross-validation
outer_splits = []
for i in range(config["n_repeats"]):
    sgkf = StratifiedGroupKFold(
        n_splits=config["outer_splits"],
        shuffle=True,
        random_state=42+i
    )
    outer_splits.extend(list(sgkf.split(X, y, groups)))

inner_cv = StratifiedGroupKFold(
    n_splits=config["inner_splits"],
    shuffle=True,
    random_state=42
)

# Setup hyperparameter tuning
search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=config["n_iter"],
    scoring="roc_auc",
    n_jobs=config["inner_n_jobs"],
    cv=inner_cv,
    verbose=config["inner_verbose"],
    random_state=42,
    refit=True,
    error_score='raise'
)

# Execute nested cross-validation
results = cross_validate(
    search,
    X=X,
    y=y,
    params={'groups': groups},
    cv=outer_splits,
    scoring=scoring,
    return_estimator=True,
    n_jobs=config["outer_n_jobs"],
    verbose=config["outer_verbose"],
    error_score='raise'
)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 128 concurrent workers.


[CV] START .....................................................................
Fitting 5 folds for each of 5 candidates, totalling 25 fits
[CV] START .....................................................................
Fitting 5 folds for each of 5 candidates, totalling 25 fits
[CV] START .....................................................................
Fitting 5 folds for each of 5 candidates, totalling 25 fits
[CV] START .....................................................................
Fitting 5 folds for each of 5 candidates, totalling 25 fits
[CV] START .....................................................................
Fitting 5 folds for each of 5 candidates, totalling 25 fits
[CV] START .....................................................................
Fitting 5 folds for each of 5 candidates, totalling 25 fits
[CV] START .....................................................................
Fitting 5 folds for each of 5 candidates, totalling 25 fits
[CV] START ..

[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:  1.0min


[CV] END  average_precision: (test=0.389) balanced_accuracy: (test=0.551) f1: (test=0.488) roc_auc: (test=0.546) total time=  57.6s


[Parallel(n_jobs=-1)]: Done   2 out of  15 | elapsed:  1.0min remaining:  6.7min


...

[Parallel(n_jobs=-1)]: Done  11 out of  15 | elapsed:  1.1min remaining:   23.0s
[Parallel(n_jobs=-1)]: Done  12 out of  15 | elapsed:  1.1min remaining:   15.8s


[CV] END  average_precision: (test=0.440) balanced_accuracy: (test=0.541) f1: (test=0.509) roc_auc: (test=0.566) total time=  59.6s


[Parallel(n_jobs=-1)]: Done  13 out of  15 | elapsed:  1.1min remaining:    9.8s


[CV] END  average_precision: (test=0.462) balanced_accuracy: (test=0.563) f1: (test=0.551) roc_auc: (test=0.559) total time=  59.8s
[CV] END  average_precision: (test=0.445) balanced_accuracy: (test=0.561) f1: (test=0.498) roc_auc: (test=0.567) total time= 1.0min


[Parallel(n_jobs=-1)]: Done  15 out of  15 | elapsed:  1.1min remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  15 out of  15 | elapsed:  1.1min finished


In [5]:
# DataFrame of results with best parameters
results_df = pd.DataFrame(results).drop(columns=["estimator"])
params_df = pd.DataFrame.from_records(est.best_params_ for est in results["estimator"])

if config["model_name"] == "GB":
    params_df["classifier__n_estimators"] = [
        est.best_estimator_.named_steps["classifier"].n_estimators_ for est in results["estimator"]
    ]
elif config["model_name"] == "MLP":
    classifiers = [
        est.best_estimator_.named_steps["classifier"]
        for est in results["estimator"]
    ]
    params_df["classifier__loss"] = [
        getattr(clf, "loss_", None)
        for clf in classifiers
    ]
    params_df["classifier__best_loss"] = [
        getattr(clf, "best_loss_", None)
        for clf in classifiers
    ]
    params_df["classifier__best_validation_score"] = [
        getattr(clf, "best_validation_score_", None)
        for clf in classifiers
    ]
    params_df["classifier__n_iter"] = [
        getattr(clf, "n_iter_", None)
        for clf in classifiers
    ]

results_df = pd.concat([results_df.reset_index(drop=True), params_df.reset_index(drop=True)], axis=1)
results_df = results_df.sort_values("test_roc_auc", ascending=False)
# results_df.to_csv(f"{out_dir}/results.csv", index=False)

# Summary statistics of metrics
scoring_statistics_df = pd.DataFrame({
    k: results[f"test_{v}"] for k, v in scoring.items()
}).agg(["mean", "std"]).T
scoring_statistics_df.to_csv(f"{out_dir}/scoring_statistics.csv", index=True)

# Outer CV results
n_outer = config["outer_splits"]
n_total = config["outer_splits"] * config["n_repeats"]
outer_df = pd.DataFrame({
    "repeat": (np.arange(n_total) // n_outer) + 1,
    "outer_fold": (np.arange(n_total) % n_outer) + 1,
    **{k: results[f"test_{v}"] for k, v in scoring.items()}
})
# outer_df.to_csv(f"{out_dir}/outer_cv_results.csv", index=False)

if config["model_name"] != "MLP":
    inner_df = pd.DataFrame([
        {
            "repeat": (i // config["outer_splits"]) + 1,
            "outer_fold": (i % config["outer_splits"]) + 1,
            "inner_best_score": est.best_score_,
            "inner_best_params": est.best_params_,
            "selected_features": list(est.best_estimator_.named_steps["feature_selector"].get_feature_names_out()) if config["feature_selector_method"] != "passthrough" else "passthrough",
            "n_selected_features": len(list(est.best_estimator_.named_steps["feature_selector"].get_feature_names_out())) if config["feature_selector_method"] != "passthrough" else "passthrough"
        }
        for i, est in enumerate(results["estimator"])
    ])
    # inner_df.to_csv(f"{out_dir}/inner_cv_results.csv", index=False)

else:
    inner_df = pd.DataFrame([
        {
            "repeat": (i // config["outer_splits"]) + 1,
            "outer_fold": (i % config["outer_splits"]) + 1,
            "inner_best_score": est.best_score_,
            "inner_best_params": est.best_params_,
            "selected_features": list(est.best_estimator_.named_steps["feature_selector"].get_feature_names_out()) if config["feature_selector_method"] != "passthrough" else "passthrough",
            "n_selected_features": len(list(est.best_estimator_.named_steps["feature_selector"].get_feature_names_out())) if config["feature_selector_method"] != "passthrough" else "passthrough"
        }
        for i, est in enumerate(results["estimator"])
    ])

    loss_validation_df = pd.DataFrame([
        {
            "repeat": (i // config["outer_splits"]) + 1,
            "outer_fold": (i % config["outer_splits"]) + 1,
            "iteration": iteration + 1,
            "loss": loss,
            "validation_score": score
        }
        for i, clf in enumerate(classifiers)
        for iteration, (loss, score) in enumerate(zip(getattr(clf, "loss_curve_", []), getattr(clf, "validation_scores_", [])))
    ])
    
    # inner_df.to_csv(f"{out_dir}/inner_cv_results.csv", index=False)
    # loss_validation_df.to_csv(f"{out_dir}/loss_validation_curves.csv", index=False)

# if config["feature_selector_method"] == "corr_based":
    #     inner_df = pd.DataFrame([
    #         {
    #             "repeat": (i // config["outer_splits"]) + 1,
    #             "outer_fold": (i % config["outer_splits"]) + 1,
    #             "inner_best_score": est.best_score_,
    #             "inner_best_params": est.best_params_,
    #             "selected_features": list(est.best_estimator_.named_steps["feature_selector"].selected_features_),
    #             "n_selected_features": len(list(est.best_estimator_.named_steps["feature_selector"].selected_features_))
    #         }
    #         for i, est in enumerate(results["estimator"])
    #     ])
    # else:

In [5]:
all_shap_dfs, total_shap_df, shap_df_avg = compute_fold_shap(outer_splits, results, model_name, X, y, config)

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

...

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

In [6]:
plot_shap_summary(shap_df_avg, X, config)

### High-level pipeline

In [13]:
%load_ext autoreload
%autoreload 2

from scripts.utils.data_loader import create_configs
from scripts.utils.train_tune_val import run_experiment
from scripts.utils.model_factory import param_space

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# Setup experiment configurations
case_idx = 8
model_name = "MLP"
feature_selector_method = "rfe"

N_REPEATS = 7
OUTER_SPLITS = 5
INNER_SPLITS = 5
N_ITER = 60
TOTAL_OUTER_FITS = N_REPEATS * OUTER_SPLITS
ALLOCATED_CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1))

n_dict = {
    "n_repeats": N_REPEATS,
    "outer_splits": OUTER_SPLITS,
    "inner_splits": INNER_SPLITS,
    "n_iter": N_ITER,
    "outer_verbose": 20,
    "inner_verbose": 1,
    "outer_n_jobs": min(ALLOCATED_CPUS, TOTAL_OUTER_FITS),
    "inner_n_jobs": 1
}

config = create_configs(case_idx, model_name, feature_selector_method, n_dict)
out_dir = f"../results_tests/{model_name}_{feature_selector_method}/sex={config['sexes_key']}/task={config['tasks_key']}"
config.update({"out_dir": out_dir})
os.makedirs(out_dir, exist_ok=True)

# Run experiment
run_experiment(config)